In [ ]:
!pip install -r requirements.txt


In [1]:
import argparse
import pandas as pd
import os
import json
import torch
import torch.nn as nn
import numpy as np
from finetune_data import train_multi_seed,loadData


In [9]:
import importlib
import finetune_data
importlib.reload(finetune_data)
from finetune_data import train_multi_seed, loadData

In [2]:
args = argparse.Namespace()

# Set dataset
args.dataset = "BBC_News"
args.report_per_epoch= True

# Set training configurations.
training_configs = \
{
    "task": "binary",   # task should match your choosen dataset
    "num_of_seeds": 1, # would use multiple seeds with more resources
    "initial_seed": 123,
    "epochs_per_seed": 2,   # we only train 2 epochs in this demo to reduce training time. that means the model sees the data twice, updating as it goes
    "train_batch_size": 16, # can try 8 or 32, it controls the amount of data used to update the model per update
    "max_seq_length": 128, # don't change this one, these models use smaller sequences, some new models can use larger sequences. the sequence is where the transformer operates
    "models": [             # We include one model in this example. You can include as many as you like.
        {
            "model_name": "ConfliBERT-scr-uncased",
            "model_path": "snowood1/ConfliBERT-scr-uncased", # simpletransformers gets this model from huggingface
            "architecture": "bert",
            "do_lower_case": True # should be true if using an uncased model
        } # , this part is commented out so it won't execute, but this is how you'd add additional models
        #{
        #    "model_name": "bert-base-uncased",
        #    "model_path": "bert-base-uncased",
        #    "architecture": "bert",
        #    "do_lower_case": True
        #}
    ]
}

## Saving the custom configuration.
for k,v in training_configs.items():
    setattr(args, k, v)

args.data_dir = os.path.join("./data/", args.dataset, "")

## Loading the datasets. loadData is a function in finetune_data.py
train_df, eval_df, test_df, args.num_labels = loadData(args)

if args.task == "ner":
    with open(os.path.join(args.data_dir, "labels.json")) as json_file:
        args.labels_list = json.load(json_file)

In [ ]:
import sys
!{sys.executable} -m pip install 'accelerate>=1.1.0'

Looking in indexes: https://pypi.org/simple, https://packagecloud.io/github/git-lfs/pypi/simple


In [3]:
## Running experiments for all the models in configs:
for model_configs in args.models:

    # args.output_dir = os.path.join("./outputs/", args.dataset + "_" + model_configs["model_name"], "")
    args.output_dir = os.path.join("./outputs/", args.dataset, "")

    train_multi_seed(args, train_df, eval_df, test_df, model_configs)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: snowood1/ConfliBERT-scr-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if 

Epoch,Training Loss,Validation Loss,F1
1,0.108887,0.042494,0.984127
2,0.027808,0.004146,0.992248


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]